
PyTorch `torch.nn` 模块

本文按“单层 → 多层 → 常用网络组件 → 模型参数与模式管理 → 完整练习”的顺序，系统学习 `torch.nn`。

在学习层之前，先熟悉张量形状。常见记号：`N` 表示批次大小，`D` 表示特征数，`C` 表示类别或通道数，`L` 表示序列长度，`V` 表示词表大小。`nn` 层通常不会自动猜测维度语义，输入形状必须符合该层约定。

In [3]:
import torch
from torch import nn

In [5]:
# 2. 网络的基本单位：`nn.Module`
class AffineModel(nn.Module):

    # - `__init__()`：创建并注册子模块、参数和 buffer。记得调用 `super().__init__()`。
    def __init__(self, in_features, out_features):
        super(AffineModel, self).__init__()
        # 输入4个特征，输出2个特征
        self.affine = nn.Linear(in_features=in_features, out_features=out_features)

    # - `forward(x)`：描述一次前向计算，即输入如何经过各层得到输出。
    def forward(self, x):
        return self.affine(x)

model = AffineModel(in_features=4, out_features=2)
# 3个数据，每个数据4个特征
input = torch.randn(3, 4)
# 前向传播
# - `model(x)`：推荐调用方式。它会经过 `nn.Module` 的调用逻辑，包括 hooks 等机制，再进入 `forward()`。
# 一般不要直接写 `model.forward(x)`，否则会绕过模块调用入口上处理的一些机制。
output = model(input)
print(output)

tensor([[ 0.7945, -0.3369],
        [ 0.2628,  0.4578],
        [ 0.4578,  0.6198]], grad_fn=<AddmmBackward0>)


In [6]:
def device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')
device = device()
device

device(type='mps')

In [9]:
x = torch.randn(3, 4)
x1 = x.to(device)
print(x1)
x2 = x.to(torch.device('cpu'))
print(x2)

tensor([[-0.4289,  0.6422, -1.3049,  0.4662],
        [ 0.9108, -1.2964,  0.0953,  1.7659],
        [-0.6900,  0.4526, -0.2505, -0.3310]], device='mps:0')
tensor([[-0.4289,  0.6422, -1.3049,  0.4662],
        [ 0.9108, -1.2964,  0.0953,  1.7659],
        [-0.6900,  0.4526, -0.2505, -0.3310]])


In [16]:
# 权重 `W` 的形状为 `[out_features, in_features]`，偏置 `b` 的形状为 `[out_features]`。默认 `bias=True`。
layer = nn.Linear(in_features=4, out_features=2)
x = torch.randn(3, 4)
y = layer(x)
print(layer.weight.shape)
print(layer.bias.shape)
print(y.shape)


torch.Size([2, 4])
torch.Size([2])
torch.Size([3, 2])


In [18]:
# 批次维度不需要逐个循环处理：`Linear` 会对最后一维变换，并保留所有前置维度：
# nn.Linear 只变换输入张量的最后一维，前面的维度全部保留，因此不需要手动循环处理批次和序列位置。
layer = nn.Linear(8, 5)
x = torch.randn(2, 7, 8)  # 2 个样本，每个样本 7 个位置，每个位置 8 个特征
y = layer(x)
print(y.shape)  # [2, 7, 5]
# [2, 7, 8] ---最后一维的数据输入layer经过转化之后8维的输入特征转化为了5维的输出特征----> [2, 7, 5], 前面两个维度的2，7不变
# 如果输入为 `[N, D]`，则 `in_features` 必须等于 `D`。如果出现 `mat1 and mat2 shapes cannot be multiplied`，先打印输入 `shape` 和层的 `in_features`，通常是最后一维不匹配。

# 对于 `nn.Linear(4, 2)`，输入 `[4]` 会得到 `[2]`；输入 `[N, 4]` 会得到 `[N, 2]`。数据加载器一般提供批次形式 `[N, D]`，保持批次维度能让训练和推理代码更统一。

torch.Size([2, 7, 5])


In [21]:
# 4. 激活函数：给网络加入非线性
# 4.2 `nn.GELU`,  ReLU=max(0,x)，小于0的值设置为0，大于0的值不变
# ReLU 常用于 MLP、CNN 的隐藏层。默认 `inplace=False`；初学时优先保持非原地操作，以减少与 autograd 原地修改规则相关的问题。

relu = nn.ReLU()
x = torch.tensor([-2, -1, 0, 1, 2], dtype=torch.float32)
y = relu(x)
print(y)


tensor([0., 0., 0., 1., 2.])


In [23]:
# # 4.2 `nn.GELU`
# GELU 是平滑的门控型激活，常见于 Transformer：
gelu = nn.GELU()
x = torch.tensor([-2, -1, 0, 1, 2], dtype=torch.float32)
print(gelu(x))

tensor([-0.0455, -0.1587,  0.0000,  0.8413,  1.9545])


In [24]:
# 4.3 `nn.Sigmoid`, Sigmoid 将数值映射到 `(0, 1)`：
# 常用于需要概率形式的二分类输出或门控结构
sigmoid = nn.Sigmoid()
x = torch.tensor([-2, -1, 0, 1, 2], dtype=torch.float32)
print(sigmoid(x))

tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])


In [30]:
# 4.4 `nn.Softmax`
# Softmax 沿指定维度把一组分数转换为和为 1 的概率：
softmax = nn.Softmax(dim=0)
x = torch.tensor([-2, -1, 0, 1, 2], dtype=torch.float32)
pro = softmax(x)
print(pro.shape) # 输入是多少个数据，输出还是多少个数据
print(pro.sum(dim=0)) # 概率和为1

torch.Size([5])
tensor(1.0000)


In [32]:
# 4.5 激活层的常见排列
# 常见 MLP 隐藏层顺序为“线性变换 → 激活”：
block = nn.Sequential(
    nn.Linear(16, 32),
    nn.ReLU(),
)

# 最后一层是否使用激活取决于任务。回归通常输出原始值；多分类通常输出 logits；二分类搭配 `BCEWithLogitsLoss` 时也输出原始 logits。
# logits 指的是模型最后一层输出的、尚未经过 Sigmoid 或 Softmax 等概率转换的原始分数。 你可以先把它理解成：模型对每个类别的“打分”，分数越高，模型通常越倾向于预测该类别。
# 注意 logits 就是分数，并不是概率，logits 输出的数值是会大于1的，要想转化为概率那么必须使用Softmax将其转化
#  logits 也就是经过某个层输出的数据


In [35]:
# 5. 多层网络：`nn.Sequential` 与自定义 `forward()`
# 5.1 `nn.Sequential`
# 当数据只需按固定顺序依次经过各层，可用 `nn.Sequential` 快速搭建：
model = nn.Sequential(
    nn.Linear(10, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 3),
)

x = torch.randn(5, 10)
logits = model(x)
print(logits.shape)  # [5, 3]

# 数据依次流过每个子模块。`Sequential` 会注册其中的模块和参数，适合线性堆叠结构。
# 带名称时可以使用 `OrderedDict`，方便在打印模型或读取状态字典时识别层：
from collections import OrderedDict

model = nn.Sequential(OrderedDict([
    ("hidden", nn.Linear(4, 16)),
    ("activation", nn.ReLU()),
    ("classifier", nn.Linear(16, 3)),
]))


torch.Size([5, 3])


In [92]:
# 自定义一个MLP，多层感知机，使用 nn.Sequential 序列化定义
class MLP(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features):
        super(MLP, self).__init__()
        self.sequential = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, int(hidden_dim / 2)),
            nn.ReLU(),
            nn.Linear(int(hidden_dim / 2), out_features),
        )
        # self.fc = nn.Linear(in_features=in_features, out_features=hidden_dim)
        # self.activation = nn.ReLU()
        # self.fc2 = nn.Linear(in_features=hidden_dim, out_features=int(hidden_dim / 2))
        # self.classifier = nn.Linear(in_features=int(hidden_dim / 2), out_features=out_features)

    def forward(self, x):
        return self.sequential(x)
        # x = self.fc(x)
        # x = self.activation(x)
        # x = self.fc2(x)
        # x = self.activation(x)
        # x = self.classifier(x)
        # return x

# 32是批次大小， 784是每一张图片的特征数量（图片展开之后的总特征数量，28*28=784）
images = torch.randn(32, 784)
model = MLP(in_features=784, hidden_dim=256, out_features=10)
logites = model(images)
print(logites.shape)
# 输入             [N, 784]
# Linear(784, 256) [N, 256]
# ReLU             [N, 256]
# Linear(256, 128) [N, 128]
# GELU             [N, 128]
# Linear(128, 10)  [N, 10]



# 网络的参数管理
# `parameters()` 返回递归注册的参数，常传给优化器；
# `named_parameters()` 同时返回每个参数的路径名称：
print("parameters(): ")
for parameter in model.parameters():
    print(parameter.shape)

print("named_parameters(): ")
for name, parameter in model.named_parameters():
    print(name, parameter.shape, parameter.requires_grad)

# 参数数量的统计
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(total, trainable)

# `state_dict()` 状态字典：返回模型的可学习参数和持久化 buffer，键通常按模块路径命名：
for key, value in model.state_dict().items():
    print(key, value.shape)



torch.Size([32, 10])
parameters(): 
torch.Size([256, 784])
torch.Size([256])
torch.Size([128, 256])
torch.Size([128])
torch.Size([10, 128])
torch.Size([10])
named_parameters(): 
sequential.0.weight torch.Size([256, 784]) True
sequential.0.bias torch.Size([256]) True
sequential.2.weight torch.Size([128, 256]) True
sequential.2.bias torch.Size([128]) True
sequential.4.weight torch.Size([10, 128]) True
sequential.4.bias torch.Size([10]) True
235146 235146
sequential.0.weight torch.Size([256, 784])
sequential.0.bias torch.Size([256])
sequential.2.weight torch.Size([128, 256])
sequential.2.bias torch.Size([128])
sequential.4.weight torch.Size([10, 128])
sequential.4.bias torch.Size([10])


In [43]:
# 5.4 重复结构：`ModuleList` 和 `ModuleDict`
# 动态循环调用一组层时用 `nn.ModuleList`，而不是普通 `list`：
class RepeatedMLP(nn.Module):
    def __init__(self, width, depth):
        super(RepeatedMLP, self).__init__()
        self.hidden = nn.ModuleList([
            nn.Linear(width, width) for _ in range(depth)
        ])
        self.activation = nn.ReLU()
    def forward(self, x):
        for layer in self.hidden:
            x = self.activation(layer(x))
        return x
# `ModuleList` 负责注册模块，但不会像 `Sequential` 那样自动定义前向顺序，需要自行在 `forward()` 中迭代。


In [50]:
# 多个命名分支可用 `nn.ModuleDict`：
class MultiTaskHead(nn.Module):
    def __init__(self, hidden_dim):
        super(MultiTaskHead, self).__init__()
        self.heads = nn.ModuleDict({
            "classifier": nn.Linear(hidden_dim, 10),
            "regressor": nn.Linear(hidden_dim, 1),
        })

    def forward(self, x, task):
        return self.heads[task](x)

x = torch.randn(2, 32)
model = MultiTaskHead(32)
print(model(x, task="classifier"))
print(model(x, task="regressor"))


### 9.4 `nn.ModuleList`、`nn.ModuleDict` 与 `Sequential` 的选择

# | 结构 | 自动注册子模块 | 自动按顺序执行 | 适用情形 |
# | --- | --- | --- | --- |
# | `nn.Sequential` | 是 | 是 | 固定的简单串行网络 |
# | `nn.ModuleList` | 是 | 否 | 需要在循环或自定义逻辑中逐层调用 |
# | `nn.ModuleDict` | 是 | 否 | 按名称选择多个层、分支或任务头 |
# | Python `list` / `dict` | 否 | 否 | 不应直接用于需要模型管理的子模块 |


tensor([[ 0.2401,  0.2320,  0.0708, -0.1799,  0.5105, -0.7445, -0.4716,  0.0109,
          0.2590, -0.7427],
        [-0.6387, -0.4086, -0.5364,  0.0491, -0.4350,  0.4318, -0.3652, -0.7122,
         -0.0493,  0.6672]], grad_fn=<AddmmBackward0>)
tensor([[ 0.3600],
        [-0.5520]], grad_fn=<AddmmBackward0>)


In [ ]:
## 6. 正则化组件：`nn.Dropout`
# Dropout 在训练时按概率随机将一部分激活置零，并对保留值进行缩放。它可减少神经网络对个别特征的过度依赖。
# 训练时，Dropout 会随机将一部分神经元的输出置为 0；评估和推理时，它会自动关闭。
# p=0.5表示训练时每个元素有50%的概率被置为0，p一般从 0.1 或 0.2 开始尝试，根据验证集表现调整。
# 在 PyTorch 中，Dropout 不只是将部分值置为 0。默认还会将保留下来的值除以 \(1-p\)，这种方式叫做 Inverted Dropout。
# 例如 p=0.5 时：
# - 被选中的元素变成 0。
# - 没有被选中的元素乘以 2。


dropout = nn.Dropout(p=0.5)
x = torch.ones(4, 8)
print(x)

dropout.train()
y_train = dropout(x)   # 训练模式会随机置零并缩放
print(y_train)

dropout.eval()
y_eval = dropout(x)    # 评估模式下等同于恒等映射
print(y_eval)

# Dropout 的主要思路是：
# 1. 每次训练随机屏蔽部分神经元输出。
# 2. 让模型不能过度依赖某几个固定的特征或神经元。
# 3. 促使模型学习更稳健的特征组合。
# 4. 降低对训练数据的过度拟合风险。

# Dropout 在神经网络中应该放在 激活函数后面
seq = nn.Sequential(
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(16, 10),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(10, 5),
)



In [58]:
## 7. 归一化组件
# 归一化层根据指定维度上的统计量调整激活，有助于改善优化过程。不同归一化对哪些维度计算均值和方差并不相同。
### 7.1 `nn.BatchNorm1d`
# `BatchNorm1d(num_features)` 常用于全连接特征 `[N, C]`，也支持形状 `[N, C, L]` 的一维序列特征。它对每个通道/特征维使用批次统计量进行归一化：



In [64]:
## 8. 嵌入层：`nn.Embedding`
# Embedding 是一个可训练的查找表，将离散整数索引映射到稠密向量。若词表大小为 `V`、嵌入维度为 `D`，其权重矩阵形状为 `[V, D]`。
# 嵌入维度即位每一个词占有多少维的向量
emebdding = nn.Embedding(num_embeddings=10, embedding_dim=10)
# 索引值必须满足 `0 <= id < num_embeddings`，通常 dtype 为 `torch.long`。
token_ids = torch.tensor([1,2,3], dtype=torch.long) # 必须使用long类型的数据作为索引
embedded = emebdding(token_ids)
print(embedded)
embedded.shape

tensor([[ 0.3463, -0.0187,  0.8889, -0.9173,  0.5595, -0.2043, -1.0381,  1.2516,
         -2.4141,  0.6005],
        [-0.4912,  1.8922, -0.6663,  0.6140, -0.5037,  0.5958, -1.2033, -0.8392,
          0.1883,  1.7514],
        [-0.9139,  1.3961, -0.3779,  2.0017,  1.0763, -1.1327,  1.5759,  0.0816,
          0.0400,  0.5159]], grad_fn=<EmbeddingBackward0>)


torch.Size([3, 10])

In [67]:
### 8.1 Padding 索引
# 自然语言批处理常用 padding 填充不同长度的序列，可用 `padding_idx` 指定填充值索引：
# `padding_idx` 对应行不会按普通 Embedding 梯度方式更新，通常初始化为零。
# 注意，Embedding 只负责索引到向量，不负责分词、位置编码、padding mask 或序列模型计算。
PAD_ID = 0
embedding = nn.Embedding(num_embeddings=10, embedding_dim=10, padding_idx=PAD_ID)
token_ids = torch.tensor([0, 1, 2], dtype=torch.long)
embedded = embedding(token_ids)
print(embedded)

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000],
        [-0.1056,  2.2486, -1.4601, -0.3759,  0.4045,  0.8115,  0.8622,  0.4281,
          0.0122, -0.7222],
        [-0.3045,  1.9465, -0.1332, -0.1241, -0.6930,  0.5942,  0.1393,  0.9609,
         -1.4630, -0.5868]], grad_fn=<EmbeddingBackward0>)


In [69]:
### 8.2 Embedding 与 one-hot 的关系
# 从数学上看，查找 embedding 等价于 one-hot 向量乘以 `[V, D]` 的嵌入矩阵，但实际使用 `nn.Embedding` 无需显式构造巨大的 one-hot 张量，内存和计算更高效。


In [80]:
### 9.1 展平层 `nn.Flatten`
# 图像输入通常是 `[N, C, H, W]`，进入全连接层前可保留批次维并展平其余维，会保留第0维（最外面批次的那一个维度），其余维度会被展开，因为进入线性全连接层的时候必须是1*n的向量
# N 表示批次，C 表示通道数，H 表示高度， W 表示宽度
flatten = nn.Flatten(start_dim=1)
input = torch.randn((8, 3, 28, 28))
flattened_input = flatten(input)
print(flattened_input.shape)

torch.Size([8, 2352])


In [87]:
# ### 9.2 卷积层 `nn.Conv2d`
# # 卷积层通过局部感受野提取空间特征，输入布局通常为 `[N, C_in, H, W]`：
x = torch.randn((8, 1, 28, 28))
conv = nn.Conv2d(
    in_channels=1,   # 表示输入通道数是1
    out_channels=16, # 表示输出通道数是16
    kernel_size=3,   # 卷积核大小是3
    stride=1,        # 卷积每次移动的步幅是1步
    padding=1,       # 图像边缘填充一圈全0像素
)
y = conv(x)
# in_channels=1，out_channels=16 一个特征图的输入会产生16张特征图的输出
print(y.shape)  # torch.Size([1, 16, 28, 28])
#
# ### 9.3 池化层
# kernel_size=2 池化窗口大小为2
# stride=2 池化步幅是2
pool = nn.MaxPool2d(kernel_size=2, stride=2)
pooled_y = pool(y)
print(pooled_y.shape) # torch.Size([8, 16, 14, 14])

# # 卷积层 nn.Conv2d：提取特征。
# # 池化层 nn.MaxPool2d：压缩特征图的空间尺寸，保留局部显著特征。
# # 一般情况下经过卷积之后要经过一个池化，进而压缩特征空间

torch.Size([8, 16, 28, 28])
torch.Size([8, 16, 14, 14])


In [94]:
model = MLP(in_features=784, hidden_dim=256, out_features=10)
# 保存模型
torch.save(model, "mlp_model.pth")
# 仅保存模型参数
torch.save(model.state_dict(), "mlp_model_state_dict.pth")

# 加载模型，weights_only=False 允许反序列化完整模型对象
load_mlp_model = torch.load("mlp_model.pth", map_location=torch.device('cpu'), weights_only=False)

# 注意：只保存模型权重适合推理；要无缝续训，还要保存 optimizer 和 scheduler 状态。

# 加载模型参数
load_mlp_model2 = MLP(in_features=784, hidden_dim=256, out_features=10)
# 读取参数字典，weights_only=True 表示只加载参数
model2_state_dict = torch.load("mlp_model_state_dict.pth", map_location=torch.device("cpu"), weights_only=True)
# 将参数加载到模型中。load_state_dict 接收的是模型参数字典，而不是文件路径字符串，因此必须先使用 torch.load 来家长模型参数
load_mlp_model2.load_state_dict(model2_state_dict)
# 切换到推理模式
load_mlp_model2.eval()

MLP(
  (sequential): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [ ]:
#